# 32 — Synovitis / Effusion discordant-case audit (CPU)

Diagnostic only: NO training, NO GPU, NO submission. Attach only the RSNA competition. Predictions from experiments 26/30 are embedded; no output ZIP or model input is required. Use Accelerator None, Internet Off, Save & Run All.

This notebook checks the 58 gold identities and prepares the 18 discordant cases (5 Synovitis-only; 13 Effusion-only), plus all-case scores and a blank manual review worksheet. Scores are not calibrated diagnostic thresholds. Ranks use all 58 reference studies and are not hidden-test submission ranks. Gold was reused upstream and is NOT independent validation.

Privacy: raw reports are NOT written to output files. SHOW_REPORTS=False by default. After the saved run, you may set SHOW_REPORTS=True and run the last cell interactively to read reports inside your private notebook. If you do so, do not publish/export the executed notebook with report text; clear its outputs first. No external translator or LLM API is called. Do not treat this worksheet as clinical advice or rewrite labels automatically.

Return the small Output ZIP and, if reviewed locally, your manual annotations without raw report text. A future label-source audit needs separately pinned source tables; this notebook does not reconstruct the unknown Nartaa label blend.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

SHOW_REPORTS = False
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/rsna32_case_audit')
OUT.mkdir(parents=True, exist_ok=True)
paths = [INPUT / 'rsna-knee-abnormality-detection', INPUT / 'competitions' / 'rsna-knee-abnormality-detection']
roots = [p for p in paths if (p / 'train.csv').is_file()]
if len(roots) != 1:
    raise RuntimeError('Attach exactly one RSNA competition input containing train.csv')
train = pd.read_csv(roots[0] / 'train.csv', dtype={'StudyInstanceUID': str})
if not train.StudyInstanceUID.is_unique:
    raise RuntimeError('Duplicate training StudyInstanceUID')
REF = [{'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.68918848, 'r6_effusion': 0.66231592, 'raptor_synovitis': 0.5520537, 'raptor_effusion': 0.8344146}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269', 'gold_synovitis': 1, 'gold_effusion': 0, 'r6_synovitis': 1.0, 'r6_effusion': 0.836062897, 'raptor_synovitis': 0.7165401, 'raptor_effusion': 0.7751676}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052', 'gold_synovitis': 1, 'gold_effusion': 0, 'r6_synovitis': 0.94995296, 'r6_effusion': 0.102916289, 'raptor_synovitis': 0.57890475, 'raptor_effusion': 0.7015383}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.809358537, 'r6_effusion': 0.199304844, 'raptor_synovitis': 0.7232237, 'raptor_effusion': 0.80485743}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.00252679409, 'r6_effusion': 0.00210258238, 'raptor_synovitis': 0.3950103, 'raptor_effusion': 0.37544024}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999999881, 'r6_effusion': 0.707510288, 'raptor_synovitis': 0.8494772, 'raptor_effusion': 0.8858193}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.999987006, 'r6_effusion': 0.973201586, 'raptor_synovitis': 0.5458956, 'raptor_effusion': 0.866956}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.999876142, 'r6_effusion': 0.141171609, 'raptor_synovitis': 0.4984745, 'raptor_effusion': 0.49011105}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.0236632321, 'r6_effusion': 0.547965016, 'raptor_synovitis': 0.66913843, 'raptor_effusion': 0.830011}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999996305, 'r6_effusion': 0.951694704, 'raptor_synovitis': 0.8278583, 'raptor_effusion': 0.89417577}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945', 'gold_synovitis': 1, 'gold_effusion': 0, 'r6_synovitis': 0.0117911762, 'r6_effusion': 0.082391355, 'raptor_synovitis': 0.4118078, 'raptor_effusion': 0.375673}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.999019146, 'r6_effusion': 0.754259218, 'raptor_synovitis': 0.6401347, 'raptor_effusion': 0.90977263}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.999909997, 'r6_effusion': 0.0185223697, 'raptor_synovitis': 0.5550516, 'raptor_effusion': 0.71595615}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999976754, 'r6_effusion': 0.911921936, 'raptor_synovitis': 0.77386165, 'raptor_effusion': 0.8993275}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 3.50408627e-05, 'r6_effusion': 0.00394672659, 'raptor_synovitis': 0.31434342, 'raptor_effusion': 0.28598326}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.976792395, 'r6_effusion': 0.200387452, 'raptor_synovitis': 0.4604631, 'raptor_effusion': 0.5260008}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999996424, 'r6_effusion': 0.921891634, 'raptor_synovitis': 0.7962346, 'raptor_effusion': 0.88279396}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309', 'gold_synovitis': 1, 'gold_effusion': 0, 'r6_synovitis': 0.999919891, 'r6_effusion': 0.525198056, 'raptor_synovitis': 0.6611382, 'raptor_effusion': 0.7554807}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999998808, 'r6_effusion': 0.898047421, 'raptor_synovitis': 0.72505665, 'raptor_effusion': 0.933043}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.999288023, 'r6_effusion': 0.961908765, 'raptor_synovitis': 0.55417323, 'raptor_effusion': 0.8868497}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.561016083, 'r6_effusion': 0.605584038, 'raptor_synovitis': 0.5482077, 'raptor_effusion': 0.82720184}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.000813107414, 'r6_effusion': 0.0907667887, 'raptor_synovitis': 0.50737333, 'raptor_effusion': 0.6310034}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999934673, 'r6_effusion': 0.901186716, 'raptor_synovitis': 0.6890459, 'raptor_effusion': 0.838259}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.183858782, 'r6_effusion': 0.272433421, 'raptor_synovitis': 0.45141578, 'raptor_effusion': 0.6882546}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 3.98391348e-05, 'r6_effusion': 0.00050966817, 'raptor_synovitis': 0.24957857, 'raptor_effusion': 0.09276629}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.992124736, 'r6_effusion': 0.500628509, 'raptor_synovitis': 0.68778145, 'raptor_effusion': 0.7828046}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 3.72790964e-05, 'r6_effusion': 0.018836379, 'raptor_synovitis': 0.3696495, 'raptor_effusion': 0.40470755}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999993324, 'r6_effusion': 0.96778442, 'raptor_synovitis': 0.6696644, 'raptor_effusion': 0.9137982}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999997735, 'r6_effusion': 0.981442554, 'raptor_synovitis': 0.70823324, 'raptor_effusion': 0.8927208}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.99999249, 'r6_effusion': 0.950482461, 'raptor_synovitis': 0.73930156, 'raptor_effusion': 0.9735428}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 4.3577169e-05, 'r6_effusion': 0.0293928076, 'raptor_synovitis': 0.36310172, 'raptor_effusion': 0.3917604}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999999166, 'r6_effusion': 0.97100783, 'raptor_synovitis': 0.8674256, 'raptor_effusion': 0.93392766}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.999792039, 'r6_effusion': 0.0748529032, 'raptor_synovitis': 0.61770064, 'raptor_effusion': 0.81393266}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999965191, 'r6_effusion': 0.842198543, 'raptor_synovitis': 0.5808064, 'raptor_effusion': 0.8254356}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.0141653996, 'r6_effusion': 0.154833873, 'raptor_synovitis': 0.46141565, 'raptor_effusion': 0.506809}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.999840856, 'r6_effusion': 0.840740073, 'raptor_synovitis': 0.5236994, 'raptor_effusion': 0.8056376}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.999859691, 'r6_effusion': 0.668424819, 'raptor_synovitis': 0.7232132, 'raptor_effusion': 0.69452095}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999998927, 'r6_effusion': 0.785282302, 'raptor_synovitis': 0.76676786, 'raptor_effusion': 0.8046948}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999905467, 'r6_effusion': 0.848801288, 'raptor_synovitis': 0.4757817, 'raptor_effusion': 0.6753117}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999869704, 'r6_effusion': 0.968213589, 'raptor_synovitis': 0.48151034, 'raptor_effusion': 0.7660158}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999946952, 'r6_effusion': 0.953071585, 'raptor_synovitis': 0.59616613, 'raptor_effusion': 0.9113096}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999999642, 'r6_effusion': 0.960678619, 'raptor_synovitis': 0.6942262, 'raptor_effusion': 0.8543301}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.999990225, 'r6_effusion': 0.975284036, 'raptor_synovitis': 0.58741736, 'raptor_effusion': 0.9412325}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999999285, 'r6_effusion': 0.544306251, 'raptor_synovitis': 0.7377702, 'raptor_effusion': 0.860389}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.99870193, 'r6_effusion': 0.938959458, 'raptor_synovitis': 0.5980972, 'raptor_effusion': 0.8166472}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.00392012438, 'r6_effusion': 0.03415848, 'raptor_synovitis': 0.37094468, 'raptor_effusion': 0.29994643}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 2.39487576e-06, 'r6_effusion': 0.00502494529, 'raptor_synovitis': 0.31118917, 'raptor_effusion': 0.25034845}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.999999762, 'r6_effusion': 0.958816554, 'raptor_synovitis': 0.76992, 'raptor_effusion': 0.9629425}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999999523, 'r6_effusion': 0.975658224, 'raptor_synovitis': 0.80860794, 'raptor_effusion': 0.9281311}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.992450237, 'r6_effusion': 0.874401003, 'raptor_synovitis': 0.5173314, 'raptor_effusion': 0.85587347}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999928355, 'r6_effusion': 0.920959979, 'raptor_synovitis': 0.6320071, 'raptor_effusion': 0.86044145}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.00127938914, 'r6_effusion': 0.0180953461, 'raptor_synovitis': 0.5070294, 'raptor_effusion': 0.5381719}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.999999046, 'r6_effusion': 0.430884121, 'raptor_synovitis': 0.64277494, 'raptor_effusion': 0.7871127}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037', 'gold_synovitis': 0, 'gold_effusion': 1, 'r6_synovitis': 0.187669173, 'r6_effusion': 0.618366043, 'raptor_synovitis': 0.55202514, 'raptor_effusion': 0.8235937}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.999976635, 'r6_effusion': 0.947487097, 'raptor_synovitis': 0.89424336, 'raptor_effusion': 0.9765227}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767', 'gold_synovitis': 1, 'gold_effusion': 1, 'r6_synovitis': 0.99998343, 'r6_effusion': 0.783250633, 'raptor_synovitis': 0.74508965, 'raptor_effusion': 0.9008398}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312', 'gold_synovitis': 0, 'gold_effusion': 0, 'r6_synovitis': 0.000449639163, 'r6_effusion': 0.0269404186, 'raptor_synovitis': 0.37324136, 'raptor_effusion': 0.35068393}, {'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662', 'gold_synovitis': 1, 'gold_effusion': 0, 'r6_synovitis': 0.999985218, 'r6_effusion': 0.932286715, 'raptor_synovitis': 0.67435104, 'raptor_effusion': 0.8184029}]


In [ ]:
scores = pd.DataFrame(REF).set_index('StudyInstanceUID')
if len(scores) != 58 or not scores.index.is_unique:
    raise RuntimeError('Unexpected embedded reference inventory')
gold = train.loc[train[['Synovitis', 'Effusion']].notna().all(axis=1)].set_index('StudyInstanceUID')
if set(gold.index) != set(scores.index):
    raise RuntimeError('Gold UID set differs from verified experiments 26/30')
gold = gold.loc[scores.index]
if not np.array_equal(gold[['Synovitis', 'Effusion']].values,
                      scores[['gold_synovitis', 'gold_effusion']].values):
    raise RuntimeError('Gold label identity mismatch')
probability_columns = ['r6_synovitis', 'r6_effusion', 'raptor_synovitis', 'raptor_effusion']
p = scores[probability_columns].to_numpy(float)
if not np.isfinite(p).all() or not ((p >= 0) & (p <= 1)).all():
    raise RuntimeError('Invalid embedded model probabilities')
for col in probability_columns:
    scores[col+'_rank58'] = scores[col].rank(method='average', pct=True)
scores['blend_synovitis_rank58'] = .5*scores.r6_synovitis_rank58 + .5*scores.raptor_synovitis_rank58
scores['blend_effusion_rank58'] = .5*scores.r6_effusion_rank58 + .5*scores.raptor_effusion_rank58
scores['case_group'] = np.select(
    [(scores.gold_synovitis == 1) & (scores.gold_effusion == 0),
     (scores.gold_synovitis == 0) & (scores.gold_effusion == 1),
     (scores.gold_synovitis == 1) & (scores.gold_effusion == 1)],
    ['synovitis_only', 'effusion_only', 'both_positive'], default='both_negative')
counts = scores.case_group.value_counts().to_dict()
expected = {'synovitis_only':5, 'effusion_only':13, 'both_positive':22, 'both_negative':18}
if counts != expected:
    raise RuntimeError('Discordant-case inventory changed')
cases = scores.loc[scores.case_group.isin(['synovitis_only', 'effusion_only'])].copy()
cases = cases.sort_values(['case_group', 'blend_synovitis_rank58'])
scores.to_csv(OUT / '32_all58_scores.csv', index_label='StudyInstanceUID')
cases.to_csv(OUT / '32_discordant18_scores.csv', index_label='StudyInstanceUID')
review = cases[['case_group', 'gold_synovitis', 'gold_effusion']].copy()
for col in ['report_language', 'synovitis_evidence_explicit_implied_absent_unclear',
            'effusion_evidence_explicit_implied_absent_unclear',
            'severity_or_uncertainty', 'image_review_done', 'report_gold_agreement',
            'suspected_failure_mechanism', 'reviewer_notes_no_raw_report']:
    review[col] = ''
review.to_csv(OUT / '32_manual_review_template.csv', index_label='StudyInstanceUID')
summary = {'experiment':'32_synovitis_effusion_case_audit', 'status':'COMPLETE',
           'gold_studies':58, 'case_counts':counts, 'discordant_cases':len(cases),
           'training_started':False, 'submission_created':False, 'raw_reports_exported':False,
           'gold_is_independent_validation':False,
           'note':'Manual review required. No new labels, fitted weights, residual-based prediction changes or error diagnoses are generated.'}
(OUT / '32_audit_summary.json').write_text(json.dumps(summary, indent=2))
display(cases.round(5))
print(json.dumps(summary, indent=2))


In [ ]:
# Optional private review. Raw reports remain outside exported CSV/JSON files.
if SHOW_REPORTS:
    if 'Report' not in gold:
        raise RuntimeError('Competition train.csv has no Report column')
    print('PRIVATE REVIEW: clear this cell output before sharing or publishing the notebook.')
    for uid in cases.index:
        print('\nSTUDY', uid, 'GROUP', cases.loc[uid, 'case_group'])
        print(str(gold.loc[uid, 'Report']))
else:
    print('Raw report display disabled. Review template created; no raw reports exported.')
